In [1]:
# =========================================================
# V3 - LangChain Tool-Using Agent with Streaming
# =========================================================

import os
from datetime import datetime

from dotenv import load_dotenv

from langchain_deepseek import ChatDeepSeek
from langchain.agents import create_agent
from langchain.tools import tool

from langgraph.checkpoint.memory import InMemorySaver

from langchain_tavily import TavilySearch


# =========================================================
# 1. 加载环境变量
# =========================================================

load_dotenv(override=True)

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")


# =========================================================
# 2. 检查环境变量
# =========================================================

if not DEEPSEEK_API_KEY:
    raise ValueError(
        "未找到 DEEPSEEK_API_KEY，请检查 .env 文件"
    )

if not DEEPSEEK_BASE_URL:
    raise ValueError(
        "未找到 DEEPSEEK_BASE_URL，请检查 .env 文件"
    )

if not TAVILY_API_KEY:
    raise ValueError(
        "未找到 TAVILY_API_KEY，请检查 .env 文件"
    )


# =========================================================
# 3. 基础配置
# =========================================================

MODEL_NAME = "deepseek-v4-flash"

EXIT_WORDS = {
    "quit",
    "exit"
}


# =========================================================
# 4. 初始化 Chat Model
# =========================================================

model = ChatDeepSeek(
    model=MODEL_NAME,
    api_key=DEEPSEEK_API_KEY,
    api_base=DEEPSEEK_BASE_URL,
)


# =========================================================
# 5. 定义 Tools
# =========================================================


# ---------------------------------------------------------
# Tool 1: Calculator
# ---------------------------------------------------------

@tool
def calculator(
    operation: str,
    a: float,
    b: float
) -> float:
    """
    执行两个数字之间的基础数学运算。

    支持的 operation：
    - add：加法
    - subtract：减法
    - multiply：乘法
    - divide：除法

    当用户需要进行基础数学计算时使用此工具。
    """

    if operation == "add":
        return a + b

    elif operation == "subtract":
        return a - b

    elif operation == "multiply":
        return a * b

    elif operation == "divide":

        if b == 0:
            raise ValueError(
                "除数不能为 0"
            )

        return a / b

    else:
        raise ValueError(
            "不支持的操作，请使用 "
            "add、subtract、multiply 或 divide"
        )


# ---------------------------------------------------------
# Tool 2: Current Time
# ---------------------------------------------------------

@tool
def get_current_time() -> str:
    """
    获取当前计算机本地的日期和时间。

    当用户询问：
    - 现在几点
    - 当前时间
    - 今天日期
    - 今天几号

    等当前日期或时间相关问题时使用此工具。
    """

    now = datetime.now()

    return now.strftime(
        "%Y-%m-%d %H:%M:%S"
    )


# ---------------------------------------------------------
# Tool 3: Web Search
# ---------------------------------------------------------

web_search = TavilySearch(
    max_results=5
)


# ---------------------------------------------------------
# Tool List
# ---------------------------------------------------------

tools = [
    calculator,
    get_current_time,
    web_search
]


# =========================================================
# 6. System Prompt
# =========================================================

SYSTEM_PROMPT = """
你叫小明，是一名耐心、友好并且可靠的 AI 学习助手。

你的主要任务是帮助用户回答：
- 学习问题
- 编程问题
- 人工智能问题
- 计算机科学问题
- 一般知识问题

你可以根据用户的问题自主决定是否使用系统提供的工具。

=========================
工具使用规则
=========================

1. 如果用户要求进行数学计算，
   优先使用 calculator 工具。

2. calculator 支持：
   - add
   - subtract
   - multiply
   - divide

3. 如果用户询问：
   - 当前时间
   - 现在几点
   - 今天日期
   - 今天几号

   使用 get_current_time 工具。

4. 如果用户的问题涉及：
   - 最新信息
   - 实时信息
   - 最近新闻
   - 当前事件
   - 最近发生的事情
   - 模型自身知识可能已经过时的信息

   应优先使用 Web Search 工具查询。

5. 如果问题不需要任何工具，
   并且你可以可靠回答，
   则直接回答用户，不需要调用工具。

6. 不要为了展示工具而无意义地调用工具。

7. 不要伪造工具执行结果。

8. 如果工具执行失败，
   应明确向用户说明发生了错误。

9. 工具执行完成后，
   应根据工具返回结果生成清晰、自然的最终回答。

=========================
回答要求
=========================

1. 默认使用中文回答。

2. 专业术语可以保留英文。

3. 对初学者尽量使用简单易懂的语言。

4. 遇到复杂概念时，
   尽量配合具体例子解释。

5. 对代码问题：
   - 先解释基本思路
   - 再提供代码
   - 必要时解释关键代码

6. 如果不确定答案，
   请明确说明，不要编造信息。
"""


# =========================================================
# 7. 创建 Short-term Memory
# =========================================================

checkpointer = InMemorySaver()


# =========================================================
# 8. 创建 Agent
# =========================================================

agent = create_agent(
    model=model,
    tools=tools,
    system_prompt=SYSTEM_PROMPT,
    checkpointer=checkpointer,
)


# =========================================================
# 9. Conversation Config
# =========================================================

# 相同 thread_id 表示属于同一个对话。
# 因此 Agent 能够恢复之前的 conversation state。

config = {
    "configurable": {
        "thread_id": "main_conversation"
    }
}


# =========================================================
# 10. 启动提示
# =========================================================

print("=" * 60)
print("小明 AI Agent V3 已启动")
print("支持：普通问答 / 数学计算 / 当前时间 / 实时搜索")
print("输入 quit 或 exit 可以结束对话")
print("=" * 60)


# =========================================================
# 11. Main Conversation Loop
# =========================================================

round_number = 1


while True:

    print(
        f"\n{'=' * 15} "
        f"第 {round_number} 轮对话 "
        f"{'=' * 15}"
    )

    user_input = input(
        "\n你："
    ).strip()


    # -----------------------------------------------------
    # 判断是否退出
    # -----------------------------------------------------

    if user_input.lower() in EXIT_WORDS:

        print(
            "\n小明：会话已结束，欢迎下次再来。"
        )

        break


    # -----------------------------------------------------
    # 防止空输入
    # -----------------------------------------------------

    if not user_input:

        print(
            "\n请输入有效的问题。"
        )

        continue


    # -----------------------------------------------------
    # Agent Streaming
    # -----------------------------------------------------

    try:

        print(
            "\n小明：",
            end="",
            flush=True
        )


        for message_chunk, metadata in agent.stream(
            {
                "messages": [
                    {
                        "role": "user",
                        "content": user_input
                    }
                ]
            },
            config=config,
            stream_mode="messages",
        ):

            # message_chunk.content 可能为空
            # 例如模型正在生成 tool call 时，
            # content 可能为空字符串。

            if message_chunk.content:

                print(
                    message_chunk.content,
                    end="",
                    flush=True
                )


        # 当前回答完成后换行

        print()

        round_number += 1


    # -----------------------------------------------------
    # Error Handling
    # -----------------------------------------------------

    except Exception as e:

        print(
            f"\n\nAgent 调用失败：{e}"
        )


    print()

小明 AI Agent 已启动
输入 quit 或 exit 可以结束对话

========== 第 1 轮对话 ==========

小明：LangChain 是一个用于开发**基于大语言模型（LLM）应用**的 Python/JavaScript 框架。简单来说，它帮助你用 GPT、Claude 这类大模型，快速搭建出"能干活"的应用，而不是只停留在聊天对话层面。

## 为什么需要 LangChain？

直接用 API 调用大模型其实很简单，但真实应用往往很复杂，比如：

- 要让模型"记住"多轮对话
- 要让模型读你自己的文档、数据库
- 要让模型调用外部工具（搜索、计算器、API）
- 要让多个模型步骤串联成一条流水线

这些重复性的工作，LangChain 帮你封装好了，避免重复造轮子。

## 核心模块

| 模块 | 作用 | 比喻 |
|------|------|------|
| **Models** | 统一封装各种大模型（OpenAI、Claude 等） | 不同的"发动机" |
| **Prompts** | 管理、复用提示词模板 | 固定的"指令模板" |
| **Chains** | 把多个步骤串联成流水线 | "流水线" |
| **Memory** | 让模型记住历史对话 | 短期"记忆" |
| **Agents** | 让模型自主决定调用哪些工具 | 会"思考决策"的员工 |
| **Document Loaders / Splitters / Vectorstores** | 加载、切分、存储文档 | 你的"资料库" |
| **Retrievers** | 检索相关资料喂给模型（RAG） | 资料"检索员" |

## 一个直观例子

假设你想做一个"公司内部知识库问答机器人"：

```python
from langchain_community.document_loaders import TextLoader
from langchain.text_splitter import CharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FA